# Notebook 04 - SFT With Adaption-Improved Data

This is the adapted-data SFT condition. It trains the same base model with the same training settings as Notebook 03, but uses `data/processed/kakugo_adapted_sft.jsonl`.

Keeping these notebooks parallel makes the comparison easier to reason about.

**Files read:**
- [`../configs/tinker_sft_adapted.yaml`](../configs/tinker_sft_adapted.yaml) - base model, renderer, training hyperparameters, input path, and output path for the adapted-data SFT run.
- [`../data/processed/kakugo_adapted_sft.jsonl`](../data/processed/kakugo_adapted_sft.jsonl) - chat-format adapted SFT examples prepared in Notebook 02.

**Files written:**
- [`../results/models/sft_adapted/`](../results/models/sft_adapted/) - Tinker logs, metrics, checkpoints, and final model/sampler references for the adapted-data SFT run.

In [1]:
import asyncio
import json
import logging
import os
import sys
import warnings
from pathlib import Path

import nest_asyncio
from huggingface_hub import login
from tinker_cookbook.supervised.data import FromConversationFileBuilder
from tinker_cookbook.supervised.train import Config, main
from tinker_cookbook.supervised.types import ChatDatasetBuilderCommonConfig

nest_asyncio.apply()
warnings.filterwarnings('ignore', module='tinker_cookbook')

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [3]:
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml, require_file

load_env()
adapted_sft_config = load_yaml(ROOT / 'configs/tinker_sft_adapted.yaml')

require_file(ROOT / adapted_sft_config['data_path'], 'Run Notebook 02 before launching adapted-data SFT.')
ensure_dir(ROOT / adapted_sft_config['output_dir'])

if not os.environ.get('TINKER_API_KEY'):
    raise RuntimeError('Missing TINKER_TOKEN or TINKER_API_KEY. Add it to .env before launching training.')

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Review the training config

This is the exact adapted-data SFT configuration that will be sent to Tinker in the training cell below.

In [4]:
print(json.dumps(adapted_sft_config, indent=2))

{
  "run_name": "sft_adapted",
  "base_model": "Qwen/Qwen3-8B-Base",
  "renderer_name": "role_colon",
  "data_path": "data/processed/kakugo_adapted_sft.jsonl",
  "output_dir": "results/models/sft_adapted",
  "learning_rate": "2e-4",
  "num_epochs": 3,
  "batch_size": 4,
  "lora_rank": 16,
  "max_length": 2048,
  "train_on_what": "last_assistant_message",
  "save_every": 50
}


## Build the Tinker SFT config

This mirrors Notebook 03. The only intentional difference is the data path and output directory from `configs/tinker_sft_adapted.yaml`.

In [5]:
renderer_name = adapted_sft_config['renderer_name']

common_config = ChatDatasetBuilderCommonConfig(
    model_name_for_tokenizer=adapted_sft_config['base_model'],
    renderer_name=renderer_name,
    max_length=int(adapted_sft_config['max_length']),
    batch_size=int(adapted_sft_config['batch_size']),
    train_on_what=adapted_sft_config['train_on_what'],
)

dataset_builder = FromConversationFileBuilder(
    file_path=str(ROOT / adapted_sft_config['data_path']),
    common_config=common_config,
)

sft_config = Config(
    log_path=str(ROOT / adapted_sft_config['output_dir']),
    model_name=adapted_sft_config['base_model'],
    dataset_builder=dataset_builder,
    learning_rate=float(adapted_sft_config['learning_rate']),
    lora_rank=int(adapted_sft_config['lora_rank']),
    num_epochs=int(adapted_sft_config['num_epochs']),
)

print('\nSFT Config:')
print(f"  Run:           {adapted_sft_config['run_name']}")
print(f"  Model:         {sft_config.model_name}")
print(f"  Renderer:      {renderer_name}")
print(f"  Data:          {ROOT / adapted_sft_config['data_path']}")
print(f"  Output:        {sft_config.log_path}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  LoRA rank:     {sft_config.lora_rank}")
print(f"  Epochs:        {sft_config.num_epochs}")


SFT Config:
  Run:           sft_adapted
  Model:         Qwen/Qwen3-8B-Base
  Renderer:      role_colon
  Data:          /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_adapted_sft.jsonl
  Output:        /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_adapted
  Learning rate: 0.0002
  LoRA rank:     16
  Epochs:        3


## Launch training

This cell starts the real Tinker SFT job. If credentials, package imports, data paths, or Tinker access are wrong, the cell should fail loudly so the issue is visible.

In [6]:
print('=' * 50)
print('Starting adapted-data SFT training...')
print('Watch train_mean_nll; it should generally decrease across training.\n')

logging.getLogger('asyncio').setLevel(logging.CRITICAL)
result = asyncio.get_event_loop().run_until_complete(main(sft_config))

print('\nTraining complete!')
print(f'Result: {result}')

Starting adapted-data SFT training...
Watch train_mean_nll; it should generally decrease across training.



root:680 [INFO] Command line invocation: /Users/vivianamarquez/anaconda3/envs/adaption-kirundi-sft/lib/python3.11/site-packages/ipykernel_launcher.py -f /Users/vivianamarquez/Library/Jupyter/runtime/kernel-ab1a239d-cf4f-4e8f-b1eb-65a859b2f5c4.json
tinker_cookbook.utils.ml_log:618 [INFO] Logging to: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_adapted
tinker.lib.public_interfaces.service_client:75 [INFO] ServiceClient initialized for session 25f0ccf6-86c8-5c18-ba6b-96f9861d2691
tinker.lib.public_interfaces.service_client:159 [INFO] TrainingClient initialized for model 25f0ccf6-86c8-5c18-ba6b-96f9861d2691:train:0
tinker_cookbook.supervised.train:345 [INFO] Training for 6 batches x 3 epochs = 18 steps
tinker_cookbook.supervised.train:477 [INFO] Starting epoch 0
tinker_cookbook.supervised.train:374 [INFO] User: Ushobora kugura chips z'ibirayi hehe? Hitamwo ico kibereye cane kugira ngo usubize ikibazo haruguru. Ibiganiro: - akabari - ububiko - motherboard - 

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 0               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 0        │
│ learning_rate        │ 0.000200 │
│ num_loss_tokens      │ 1147     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2680     │
│ progress             │ 0.055556 │
│ time/finish_batch    │ 3.294268 │
│ time/get_batch       │ 0.007767 │
│ time/step            │ 3.292317 │
│ time/submit_batch    │ 0.008719 │
│ train_mean_nll       │ 2.053635 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ifaranga rya 1000 FRW riri mu bwoko ki, n'ukuri risa gute?

Assistant: Ni ideni (banknote) ry’ifaranga rya 1000 FRW. Rifise ibara ry’icyatsi c’urumuri hamwe n’utundi turango twa vy’umweru n’oranje. Ku mpande harimwo ishusho y’umukobwa w’umunyarwanda, ibiyaga vya Kivu n’urukuta rw’umusozi wa Karisimbi, hanyuma rugira ibimenyetso vyo

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 1               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 0        │
│ learning_rate        │ 0.000189 │
│ num_loss_tokens      │ 1481     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2439     │
│ progress             │ 0.111111 │
│ time/finish_batch    │ 1.283741 │
│ time/get_batch       │ 0.008337 │
│ time/step            │ 1.282489 │
│ time/submit_batch    │ 0.009299 │
│ train_mean_nll       │ 3.298687 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Nshobora kubona igitekerezo co gukora imitsi y'ibigari mu wa mugoroba w'umwiherwe no ku gicuku c'ikiruhuko, ntibigaruke mu masaha y'ukora?

Assistant: Ndagusavye igitekerezo co kuguma mu biruhuko no mu mugoroba: - Gusura parike canke ikibira wiyuhagira, wirahura n'ibisiga vy'umujyi. - Gukora isuku y'ikibanza c'iwanyu: gukata ibiti,

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 2               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 0        │
│ learning_rate        │ 0.000178 │
│ num_loss_tokens      │ 657      │
│ num_sequences        │ 4        │
│ num_tokens           │ 1862     │
│ progress             │ 0.166667 │
│ time/finish_batch    │ 3.800251 │
│ time/get_batch       │ 0.009447 │
│ time/step            │ 3.799424 │
│ time/submit_batch    │ 0.010491 │
│ train_mean_nll       │ 2.838549 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Andika fonction ya C++ isubiza inyuma urunigi rwiwe hanyuma igacapura kuri console.
"programming"

Assistant: #include <iostream> #include <string> #include <algorithm> void reverse_and_print(std::string str) { std::reverse(str.begin(), str.end()); std::cout << str << std::endl; } int main() { reverse_and_print("programming"); retu

...

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 14              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000044 │
│ num_loss_tokens      │ 1249     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2536     │
│ progress             │ 0.833333 │
│ time/finish_batch    │ 1.365135 │
│ time/get_batch       │ 0.008217 │
│ time/step            │ 1.364614 │
│ time/submit_batch    │ 0.008923 │
│ train_mean_nll       │ 2.641182 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ndashaka kumenya niba umubiri wanjye ushobora kubona intungamubiri z'ingenzi mu ifunguro ry'umuceri n'inyama. Nsobanurira mu buryo bworoshye, ungaragarize urutonde rw'intungamubiri z'ingenzi (poroteyine, ibinyamavuta, ibinyabutabire, vitamini n'iminerali), ingano yabyo mu ifunguro ry'amasahani, ndetse n'uko zifasha umubiri. Niba ar

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 15              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000033 │
│ num_loss_tokens      │ 1339     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2283     │
│ progress             │ 0.888889 │
│ time/finish_batch    │ 1.272978 │
│ time/get_batch       │ 0.007041 │
│ time/step            │ 1.272597 │
│ time/submit_batch    │ 0.008571 │
│ train_mean_nll       │ 1.682650 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ndipfuye kumenya ibiranga inyuguti 'N' na 'G' mu mbwirwaribanga y'inyandiko, niyoheko ziri mu rutonde rw'inyuguti?

Assistant: Inyuguti **N** n’iya **G** ni intungane mu ndimi ziri mu ndimi z’ikinyarwanda/ikirundi: - **N** n’inyuguti ya 14 mu rutonde rw’inyuguti. Ni inyuguti y’umunani (consonant) ikoreshejwe mu ndimi nka “nta”, “nk

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 16              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000022 │
│ num_loss_tokens      │ 1395     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2136     │
│ progress             │ 0.944444 │
│ time/finish_batch    │ 1.843667 │
│ time/get_batch       │ 0.005433 │
│ time/step            │ 1.842512 │
│ time/submit_batch    │ 0.006159 │
│ train_mean_nll       │ 2.149463 │
└──────────────────────┴──────────┘
tinker_cookbook.utils.ml_log:206 [INFO] Wrote metrics to /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_adapted/metrics.jsonl


tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 17              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000011 │
│ num_loss_tokens      │ 1002     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2646     │
│ progress             │ 1.000000 │
│ time/finish_batch    │ 1.930055 │
│ time/step            │ 1.929188 │
│ train_mean_nll       │ 2.305188 │
└──────────────────────┴──────────┘
tinker_cookbook.checkpoint_utils:466 [INFO] Saved checkpoints: {'state_path': 'tinker://25f0ccf6-86c8-5c18-ba6b-96f9861d2691:train:0/weights/final', 'sampler_path': 'tinker://25f0ccf6-86c8-5c18-ba6b-96f9861d2691:train:0/sampler_weights/final'}
tinker_cookbook.supervised.train:518 [INFO] Training completed successfully



Training complete!
Result: None
